In [4]:
import os
from pathlib import Path
os.chdir(Path.cwd().parent)
from data_processing.data_analysis import select_problem_sample_for_model 
deepseek="DeepSeek-R1"
df = select_problem_sample_for_model(deepseek)
df

d:\conda\envs\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,Unnamed: 0,source,problem,competition,unique_problem_label,correct,parsed_answer,gold_answer,output_cost_per_tokens,problem_idx,cost,output_tokens,input_tokens,answer,user_message,idx_answer,model_config,model_name,input_cost_per_tokens,ten_percentile_group
3348,1440,NaN,"Let $x_1, x_2, x_3, \ldots$ be a sequence of r...",MathArena/aime_2025_outputs,MathArena/aime_2025: 28,False,36,248,2.18,28,0.039087,17893.0,161.0,"Given the sequence \( x_1, x_2, x_3, \ldots \)...","Please reason step by step, and put your final...",0,deepseek/deepseek_r1,DeepSeek-R1,0.5,2
3392,1484,NaN,"Let $ABCDE$ be a convex pentagon with $AB=14$,...",MathArena/aime_2025_outputs,MathArena/aime_2025: 14,False,45,60,2.18,14,0.158214,72436.0,608.0,Given a convex pentagon \(ABCDE\) with side le...,"Please reason step by step, and put your final...",0,deepseek/deepseek_r1,DeepSeek-R1,0.5,1
13220,1472,NaN,Let $\triangle A B C$ be an equilateral triang...,MathArena/hmmt_feb_2025_outputs,MathArena/hmmt_feb_2025: 25,False,sqrt(11),\sqrt{23}-2 \sqrt{3},2.18,25,0.030148,13805.0,106.0,Given an equilateral triangle \( \triangle ABC...,"Please reason step by step, and put your final...",0,deepseek/deepseek_r1,DeepSeek-R1,0.5,3
13272,1524,NaN,Define $\operatorname{sgn}(x)$ to be $1$ when ...,MathArena/hmmt_feb_2025_outputs,MathArena/hmmt_feb_2025: 8,False,4/11,1-\frac{2}{\pi},2.18,8,0.029261,13398.0,107.0,To compute the infinite sum \(\sum_{n=1}^{\inf...,"Please reason step by step, and put your final...",0,deepseek/deepseek_r1,DeepSeek-R1,0.5,4
22520,972,NaN,"Alice has $10$ gifts $g_{1}, g_{2}, \ldots, g_...",MathArena/brumo_2025_outputs,MathArena/brumo_2025: 12,False,123,125,2.18,12,0.035888,16432.0,133.0,"Alice has 10 gifts and 10 friends, where each ...","Please reason step by step, and put your final...",0,deepseek/deepseek_r1,DeepSeek-R1,0.5,5


## Problem 1 - Difficulty level - 5

In [5]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==5].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

unique_problem_label                             MathArena/brumo_2025: 12
answer                  Alice has 10 gifts and 10 friends, where each ...
gold_answer                                                           125
ten_percentile_group                                                    5
problem                 Alice has $10$ gifts $g_{1}, g_{2}, \ldots, g_...
Name: 22520, dtype: object

In [6]:
import textwrap

problem_description = df_pruned["problem"]

problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=problem_description, width=80))
print("answer", problem_gold_answer)

Alice has $10$ gifts $g_{1}, g_{2}, \ldots, g_{10}$ and $10$ friends $f_{1},
f_{2}, \ldots, f_{10}$. Gift $g_{i}$ can be given to friend $f_{j}$ if  $$
i-j=-1,0, \text { or } 1 \quad(\bmod 10) $$  How many ways are there for Alice
to pair the $10$ gifts with the $10$ friends such that each friend receives one
gift?
answer 125


### Reflexion Method

In [ ]:
from multi_agent.multi_agent import Problem
from models.prompt_template import Reflexion_Solver, Reflector
problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=problem_description,
    answer=problem_gold_answer
)

print(problem)

Welcome Reflexion_Solver, and Reflector. Together, you should solve the
following problem: >> Alice has $10$ gifts $g_{1}, g_{2}, \ldots, g_{10}$ and
$10$ friends $f_{1}, f_{2}, \ldots, f_{10}$. Gift $g_{i}$ can be given to friend
$f_{j}$ if  $$ i-j=-1,0, \text { or } 1 \quad(\bmod 10) $$  How many ways are
there for Alice to pair the $10$ gifts with the $10$ friends such that each
friend receives one gift?.<<  "When you are done, you should submidt your answer
as: ANSWER: <your answer>.  No latex formatting, just the raw number/numbers or
strings at the very end.  Before you start sharing your toughts, give a little
summary of the conversation so far.  Give a list of the currently suggested
answers. Everytime you propose an aswer, check this list.  You proposal cannot
be in this this list. Try again and submit a new unique answer."


In [7]:
from models.azure_api import Client 
api_version="2024-06-01"
model_name="DeepSeek-R1"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

In [7]:
from baselines.reflexion.reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)


=== Attempt 1 ===


Alice has 10 gifts and 10 friends, where each gift \( g_i \) can be given to friends \( f_{i-1} \), \( f_i \), or \( f_{i+1} \) modulo 10. We need to find the number of ways to pair the gifts with the friends such that each friend receives one gift.

1. **Understanding the Constraints**: Each gift can be given to one of three friends (itself, the previous friend, or the next friend in a circular arrangement). This forms a circulant graph where each node (gift) is connected to itself and its two neighbors.

2. **Permutation with Restrictions**: The problem reduces to counting the number of permutations where each element can move to one of two adjacent positions or stay in place, considering the circular arrangement. This is a known combinatorial problem but requires specific counting due to the circular nature.

3. **Recurrence Relations and Known Sequences**: For linear arrangements, the number of such permutations follows a Fibonacci sequence. However, for circul

Reflexion didn't solve this problem.

### Tree of Thought

In [8]:
from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from openai import AzureOpenAI

config = ToTConfig(
    key_env_name="AZURE_OPENAI_API_KEY",
    endpoint_env_name= "AZURE_OPENAI_ENDPOINT",
    model_name=model_name,
    n_evaluate_sample=4,
    n_select_sample=4,
    n_generate_sample=4,
    steps=4,
    api_version=api_version,
    client_type=AzureOpenAI
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=problem_description, answer=problem_gold_answer)

print(ys)
print(infos)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import gpt_usage
usage = gpt_usage()
print(usage)

path d:\NLP-group-15
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
-- new_ys --: ('\n\nThe problem requires counting the number of permutations of 10 gifts to 10 friends arranged in a circle, where each gift \\( g_i \\) can be given to friends \\( f_{i-1}, f_i, \\) or \\( f_{i+1} \\) (mod 10). This corresponds to permutations where each element is displaced by at most one position. For a circular arrangement, the number of such permutations is given by the 10th Lucas number, which accounts for fixed points, adjacent swaps, and longer cycles formed by displacements of ±1. The Lucas sequence for \\( n=10 \\) yields:\n\nAnswer: 123', '\n\nThe problem requires counting the number of valid permutations where each gift \\( g_i \\) can be given to friend \\( f_j \\) if \\( i - j \\equiv -1, 0, \\) or \\( 1 \\pmod{10} \\). This forms a circular permutation problem with each element allowed to st

Tree of thought couldn't solve this either.

### Rejection-sampling method

In [8]:
from multi_agent.multi_agent import Role, Problem, conversation, rank_answer
from models.prompt_template import Solver, Rejector

from models.azure_api import Client
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

messages, raw, path = conversation(model=model, name="Deepseek-R1_chat_Problem1" ,n_steps=10, problem=problem)
rank_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)


STEP 0: 

Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize and submit

The rejector-sampling gets the correct answer in attempt 5 and also ranks the correct answer as the highest.

## Problem 2 - Difficulty level - 4

In [9]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==4].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

unique_problem_label                           MathArena/hmmt_feb_2025: 8
answer                  To compute the infinite sum \(\sum_{n=1}^{\inf...
gold_answer                                               1-\frac{2}{\pi}
ten_percentile_group                                                    4
problem                 Define $\operatorname{sgn}(x)$ to be $1$ when ...
Name: 13272, dtype: object

In [10]:
import textwrap

problem_description = df_pruned["problem"]

problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=problem_description, width=80))
print("answer", problem_gold_answer)

Define $\operatorname{sgn}(x)$ to be $1$ when $x$ is positive, $-1$ when $x$ is
negative, and $0$ when $x$ is $0$. Compute  $$ \sum_{n=1}^{\infty}
\frac{\operatorname{sgn}\left(\sin \left(2^{n}\right)\right)}{2^{n}} $$ (The
arguments to sin are in radians.)
answer 1-\frac{2}{\pi}


### Reflexion Method

In [11]:
from multi_agent.multi_agent import Problem
from models.prompt_template import Reflexion_Solver, Reflector
problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=problem_description,
    answer=problem_gold_answer
)

print(problem)

Welcome Reflexion_Solver, and Reflector. Together, you should solve the
following problem: >> Define $\operatorname{sgn}(x)$ to be $1$ when $x$ is
positive, $-1$ when $x$ is negative, and $0$ when $x$ is $0$. Compute  $$
\sum_{n=1}^{\infty} \frac{\operatorname{sgn}\left(\sin
\left(2^{n}\right)\right)}{2^{n}} $$ (The arguments to sin are in radians.).<<
"When you are done, you should submidt your answer as: ANSWER: <your answer>.
No latex formatting, just the raw number/numbers or strings at the very end.
Before you start sharing your toughts, give a little summary of the conversation
so far.  Give a list of the currently suggested answers. Everytime you propose
an aswer, check this list.  You proposal cannot be in this this list. Try again
and submit a new unique answer."


In [12]:
from models.azure_api import Client 
api_version="2024-06-01"
model_name="DeepSeek-R1"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

In [ ]:
from baselines.reflexion.reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)


=== Attempt 1 ===


To compute the infinite sum \(\sum_{n=1}^{\infty} \frac{\operatorname{sgn}\left(\sin \left(2^{n}\right)\right)}{2^{n}}\), we need to determine the sign of \(\sin(2^n)\) for each \(n\). The key steps are as follows:

1. **Understanding the Sign Function**: The \(\operatorname{sgn}(x)\) function returns \(1\) if \(x\) is positive, \(-1\) if \(x\) is negative, and \(0\) if \(x\) is zero. Since \(2^n\) is never an integer multiple of \(\pi\) (as \(\pi\) is irrational), \(\sin(2^n)\) is never zero, so \(\operatorname{sgn}(\sin(2^n))\) is either \(1\) or \(-1\).

2. **Modulo Operation**: The value of \(\sin(2^n)\) depends on \(2^n \mod 2\pi\). This reduces the problem to determining the quadrant in which \(2^n \mod 2\pi\) lies. If the angle is in \((0, \pi)\), \(\sin(2^n)\) is positive; if in \((\pi, 2\pi)\), it is negative.

3. **Doubling Map**: The sequence \(2^n \mod 2\pi\) follows a chaotic dynamical system known as the doubling map. This map is ergodic and equidistr

### Tree of Thought

In [ ]:
from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from openai import AzureOpenAI

config = ToTConfig(
    key_env_name="AZURE_OPENAI_API_KEY",
    endpoint_env_name= "AZURE_OPENAI_ENDPOINT",
    model_name=model_name,
    n_evaluate_sample=4,
    n_select_sample=4,
    n_generate_sample=4,
    steps=4,
    api_version=api_version,
    client_type=AzureOpenAI
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=problem_description, answer=problem_gold_answer)

print(ys)
print(infos)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import gpt_usage
usage = gpt_usage()
print(usage)

### Rejection-sampling method

In [ ]:
from multi_agent.multi_agent import Role, Problem, conversation, rank_answer
from models.prompt_template import Solver, Rejector

from models.azure_api import Client
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

messages, raw, path = conversation(model=model, name="Deepseek-R1_chat_Problem2" ,n_steps=10, problem=problem)
rank_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)

## Problem 3 - Difficulty level - 3

In [ ]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==3].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

In [ ]:
import textwrap

problem_description = df_pruned["problem"]

problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=problem_description, width=80))
print("answer", problem_gold_answer)

### Reflexion Method

In [ ]:
from multi_agent.multi_agent import Problem
from models.prompt_template import Reflexion_Solver, Reflector
problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=problem_description,
    answer=problem_gold_answer
)

print(problem)

In [ ]:
from models.azure_api import Client 
api_version="2024-06-01"
model_name="DeepSeek-R1"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

In [ ]:
from baselines.reflexion.reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)

### Tree of Thought

In [ ]:
from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from openai import AzureOpenAI

config = ToTConfig(
    key_env_name="AZURE_OPENAI_API_KEY",
    endpoint_env_name= "AZURE_OPENAI_ENDPOINT",
    model_name=model_name,
    n_evaluate_sample=4,
    n_select_sample=4,
    n_generate_sample=4,
    steps=4,
    api_version=api_version,
    client_type=AzureOpenAI
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=problem_description, answer=problem_gold_answer)

print(ys)
print(infos)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import gpt_usage
usage = gpt_usage()
print(usage)

### Rejection-sampling method

In [ ]:
from multi_agent.multi_agent import Role, Problem, conversation, rank_answer
from models.prompt_template import Solver, Rejector

from models.azure_api import Client
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

messages, raw, path = conversation(model=model, name="Deepseek-R1_chat_Problem3" ,n_steps=10, problem=problem)
rank_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)

## Problem 4 - Difficulty level - 2

In [ ]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==2].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

In [ ]:
import textwrap

problem_description = df_pruned["problem"]

problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=problem_description, width=80))
print("answer", problem_gold_answer)

### Reflexion Method

In [ ]:
from multi_agent.multi_agent import Problem
from models.prompt_template import Reflexion_Solver, Reflector
problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=problem_description,
    answer=problem_gold_answer
)

print(problem)

In [ ]:
from models.azure_api import Client 
api_version="2024-06-01"
model_name="DeepSeek-R1"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

In [ ]:
from baselines.reflexion.reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)

### Tree of Thought

In [ ]:
from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from openai import AzureOpenAI

config = ToTConfig(
    key_env_name="AZURE_OPENAI_API_KEY",
    endpoint_env_name= "AZURE_OPENAI_ENDPOINT",
    model_name=model_name,
    n_evaluate_sample=4,
    n_select_sample=4,
    n_generate_sample=4,
    steps=4,
    api_version=api_version,
    client_type=AzureOpenAI
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=problem_description, answer=problem_gold_answer)

print(ys)
print(infos)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import gpt_usage
usage = gpt_usage()
print(usage)

### Rejection-sampling method

In [ ]:
from multi_agent.multi_agent import Role, Problem, conversation, rank_answer
from models.prompt_template import Solver, Rejector

from models.azure_api import Client
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

messages, raw, path = conversation(model=model, name="Deepseek-R1_chat_Problem4" ,n_steps=10, problem=problem)
rank_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)

## Problem 5 - Difficulty level - 1

In [ ]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==1].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

In [ ]:
import textwrap

problem_description = df_pruned["problem"]

problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=problem_description, width=80))
print("answer", problem_gold_answer)

### Reflexion Method

In [ ]:
from multi_agent.multi_agent import Problem
from models.prompt_template import Reflexion_Solver, Reflector
problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=problem_description,
    answer=problem_gold_answer
)

print(problem)

In [ ]:
from models.azure_api import Client 
api_version="2024-06-01"
model_name="DeepSeek-R1"


client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

In [ ]:
from baselines.reflexion.reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)

### Tree of Thought

In [ ]:
from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from openai import AzureOpenAI

config = ToTConfig(
    key_env_name="AZURE_OPENAI_API_KEY",
    endpoint_env_name= "AZURE_OPENAI_ENDPOINT",
    model_name=model_name,
    n_evaluate_sample=4,
    n_select_sample=4,
    n_generate_sample=4,
    steps=4,
    api_version=api_version,
    client_type=AzureOpenAI
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=problem_description, answer=problem_gold_answer)

print(ys)
print(infos)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import gpt_usage
usage = gpt_usage()
print(usage)

### Rejection-sampling method

In [ ]:
from multi_agent.multi_agent import Role, Problem, conversation, rank_answer
from models.prompt_template import Solver, Rejector

from models.azure_api import Client
api_version = "2024-12-01-preview"
model_name="gpt-5-nano"

client = Client(
  api_version=api_version
)

model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

messages, raw, path = conversation(model=model, name="Deepseek-R1_chat_Problem5" ,n_steps=10, problem=problem)
rank_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)